# Exploratory Data Analysis

This notebook examines the processed vulnerability dataset before modeling. It compares CVSS severity, EPSS exploitation probability, CISA KEV inclusion, and CWE categories, then checks for common data quality problems. The goal is to see where severity and exploitation likelihood agree or diverge; these patterns can motivate a combined trained model, but do not by themselves demonstrate that a model will perform better.

Run from the VulnDetect project root or from this notebook's directory. The source dataset is `data/processed/vuln_dataset.parquet`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

candidate_paths = [Path('data/processed/vuln_dataset.parquet'), Path('../data/processed/vuln_dataset.parquet')]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find data/processed/vuln_dataset.parquet. Build the dataset first, or run this notebook from the VulnDetect project root/notebooks directory.')

df = pd.read_parquet(data_path)
required_columns = {'cve_id', 'cvss_v3_score', 'epss_score', 'in_kev', 'cwe_id'}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f'Dataset is missing required columns: {sorted(missing_columns)}')

df['cvss_v3_score'] = pd.to_numeric(df['cvss_v3_score'], errors='coerce')
df['epss_score'] = pd.to_numeric(df['epss_score'], errors='coerce')
df['in_kev'] = df['in_kev'].fillna(False).astype(bool)
print(f'Loaded {len(df):,} rows from {data_path}')
df.head()

## CVSS and EPSS score distributions

CVSS describes technical severity on a 0–10 scale, while EPSS estimates the probability of exploitation in the next 30 days on a 0–1 scale. Their histograms show how each signal is distributed in this dataset. A concentration of records at one end of a scale can affect how useful a simple cutoff is, so the distributions are useful context for later model design. The plots below show the observed data; their shape should be interpreted after the dataset is loaded.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df['cvss_v3_score'].dropna(), bins=np.arange(0, 10.5, 0.5), color='#4472C4', edgecolor='white')
axes[0].set(title='CVSS v3 score distribution', xlabel='CVSS score', ylabel='CVE count', xlim=(0, 10))
axes[1].hist(df['epss_score'].dropna(), bins=30, color='#ED7D31', edgecolor='white')
axes[1].set(title='EPSS score distribution', xlabel='EPSS score (0–1)', ylabel='CVE count', xlim=(0, 1))
plt.tight_layout()
plt.show()

df[['cvss_v3_score', 'epss_score']].describe(percentiles=[.25, .5, .75, .9, .95]).T

## Do KEV-listed CVEs look high on both scores?

The CISA KEV catalog marks vulnerabilities known to be exploited in the wild. For a simple comparison, this section labels CVSS scores of 7 or above as high and EPSS scores of 0.10 or above as high. These are transparent analysis cutoffs, not universal definitions of risk.

The tables report the share of KEV-listed CVEs in each score band, excluding records missing that score; the available-score count is shown alongside each share. The cross-tab makes disagreements visible: for example, a KEV vulnerability with lower CVSS but higher EPSS, or the reverse. Such cases motivate combining signals and evaluating a trained model against held-out outcomes. KEV is a useful observed exploitation marker, but overlap with KEV is not by itself proof of predictive performance or causation.

In [ ]:
CVSS_HIGH_THRESHOLD = 7.0
EPSS_HIGH_THRESHOLD = 0.10
kev = df.loc[df['in_kev']].copy()
kev['cvss_band'] = np.where(kev['cvss_v3_score'] >= CVSS_HIGH_THRESHOLD, 'High (≥ 7)', 'Low (< 7)')
kev['epss_band'] = np.where(kev['epss_score'] >= EPSS_HIGH_THRESHOLD, 'High (≥ 0.10)', 'Low (< 0.10)')

def score_band_summary(series, high_label, low_label):
    observed = series.dropna()
    counts = pd.Series({high_label: int((observed >= (CVSS_HIGH_THRESHOLD if 'CVSS' in high_label else EPSS_HIGH_THRESHOLD)).sum()),
                        low_label: int((observed < (CVSS_HIGH_THRESHOLD if 'CVSS' in high_label else EPSS_HIGH_THRESHOLD)).sum())})
    denominator = len(observed)
    return pd.DataFrame({'count': counts, 'fraction_of_scored_KEV': counts / denominator if denominator else np.nan,
                         'scored_KEV_total': denominator, 'missing_score': int(series.isna().sum())})

cvss_kev_summary = score_band_summary(kev['cvss_v3_score'], 'High CVSS (≥ 7)', 'Low CVSS (< 7)')
epss_kev_summary = score_band_summary(kev['epss_score'], 'High EPSS (≥ 0.10)', 'Low EPSS (< 0.10)')
print('CVSS among KEV-listed CVEs')
display(cvss_kev_summary)
print('EPSS among KEV-listed CVEs')
display(epss_kev_summary)

scored_kev = kev.dropna(subset=['cvss_v3_score', 'epss_score']).copy()
scored_kev['CVSS band'] = np.where(scored_kev['cvss_v3_score'] >= CVSS_HIGH_THRESHOLD, 'High', 'Low')
scored_kev['EPSS band'] = np.where(scored_kev['epss_score'] >= EPSS_HIGH_THRESHOLD, 'High', 'Low')
display(pd.crosstab(scored_kev['CVSS band'], scored_kev['EPSS band'], normalize='all').rename_axis('KEV CVSS band').style.format('{:.1%}'))

### Reading the score comparison

Use the output above to report how often KEV-listed vulnerabilities fall below each chosen score cutoff and how frequently the two signals disagree. A substantial low-CVSS/high-EPSS or high-CVSS/low-EPSS group would show that the scores capture different aspects of risk and could support a combined model hypothesis. If most records are high on both, these two scores may add less separation within this KEV sample. Either way, a fair claim that a trained model beats either score alone requires a defined target, a time-aware held-out evaluation, and metrics compared against CVSS-only and EPSS-only baselines.

## CWE category frequency

CWE values group vulnerabilities by weakness type. The table and chart below show the 20 most frequent recorded CWE categories. This describes the composition of this dataset, not the relative danger of each weakness category. Missing or unassigned CWE values are excluded from the frequency ranking and counted in the quality checks.

In [ ]:
cwe_counts = (df['cwe_id'].dropna().astype(str).str.strip()
              .loc[lambda values: values.ne('')]
              .value_counts().head(20).sort_values())
display(cwe_counts.sort_values(ascending=False).rename('CVE count').to_frame())
cwe_counts.plot(kind='barh', figsize=(9, max(4, 0.32 * len(cwe_counts))), color='#70AD47')
plt.title('Top 20 recorded CWE categories')
plt.xlabel('CVE count')
plt.ylabel('CWE category')
plt.tight_layout()
plt.show()

## Data quality checks

Missing scores reduce the number of vulnerabilities that can be compared directly. Duplicate CVE IDs can inflate counts and bias later training, so they need investigation before model development. This cell reports counts and fractions for missing CVSS and EPSS values, duplicate IDs, and missing CWE labels. The duplicate rows are flagged rather than silently removed so the source and any merge behavior can be reviewed.

In [ ]:
row_count = len(df)
quality = pd.DataFrame([
    {'issue': 'Missing CVSS v3 score', 'count': int(df['cvss_v3_score'].isna().sum())},
    {'issue': 'Missing EPSS score', 'count': int(df['epss_score'].isna().sum())},
    {'issue': 'Duplicate CVE ID rows', 'count': int(df['cve_id'].duplicated(keep=False).sum())},
    {'issue': 'Missing CWE category', 'count': int(df['cwe_id'].isna().sum())},
])
quality['fraction_of_rows'] = quality['count'] / row_count if row_count else np.nan
display(quality.style.format({'fraction_of_rows': '{:.2%}'}))

if quality.loc[quality['issue'].eq('Duplicate CVE ID rows'), 'count'].iat[0] > 0:
    display(df.loc[df['cve_id'].duplicated(keep=False)].sort_values('cve_id').head(20))
else:
    print('No duplicate CVE ID rows found.')

## EDA report summary

After running the notebook, summarize the observed CVSS and EPSS distributions, the fraction of KEV-listed CVEs below and above each stated cutoff, and the largest CWE categories. Note the missing-score rates and whether duplicate CVE IDs were found. These results establish data coverage and suggest whether CVSS and EPSS provide complementary signals; model superiority remains a hypothesis until tested against leakage-safe held-out data and single-score baselines.